In [1]:
import requests
import time
from datetime import datetime
import json
import sys
import pandas as pd
import numpy as np
from requests.adapters import HTTPAdapter
from requests.packages.urllib3.util.retry import Retry
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_absolute_percentage_error, mean_squared_error

class BitcoinPriceCollector:
    def __init__(self, output_file='btc_prices.json'):
        self.output_file = output_file
        self.session = self._setup_session()
        self.consecutive_errors = 0
        self.max_consecutive_errors = 5
        self.price_history = []
        self.prediction_history = []
        
    def _setup_session(self):
        session = requests.Session()
        retry_strategy = Retry(
            total=3,
            backoff_factor=1,
            status_forcelist=[429, 500, 502, 503, 504]
        )
        adapter = HTTPAdapter(max_retries=retry_strategy)
        session.mount("https://", adapter)
        return session

    def fetch_bitcoin_price(self):
        url = "https://api.coingecko.com/api/v3/simple/price"
        params = {
            'ids': 'bitcoin',
            'vs_currencies': 'usd',
            'include_market_cap': 'true',
            'include_24hr_vol': 'true',
            'include_24hr_change': 'true'
        }
        try:
            response = self.session.get(url, params=params, timeout=10)
            response.raise_for_status()
            data = response.json()
            
            price_data = {
                'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
                'price': data['bitcoin']['usd'],
                'market_cap': data['bitcoin']['usd_market_cap'],
                'volume_24h': data['bitcoin']['usd_24h_vol'],
                'change_24h': data['bitcoin']['usd_24h_change']
            }
            
            self.consecutive_errors = 0
            return price_data
            
        except requests.exceptions.RequestException as e:
            self.consecutive_errors += 1
            print(f"Error fetching price (attempt {self.consecutive_errors}): {e}")
            if self.consecutive_errors >= self.max_consecutive_errors:
                print("Too many consecutive errors. Exiting...")
                sys.exit(1)
            return None

    def make_prediction(self):
        if len(self.price_history) < 5:  # Need at least 5 points for meaningful prediction
            return None, None
            
        try:
            # Prepare data for ARIMA
            prices = pd.Series([p['price'] for p in self.price_history])
            
            # Fit ARIMA model
            model = ARIMA(prices, order=(1,1,1))  # Simple ARIMA(1,1,1) model
            model_fit = model.fit()
            
            # Make prediction
            prediction = model_fit.forecast(steps=1)[0]
            
            # Calculate accuracy metrics if we have previous predictions
            if len(self.prediction_history) > 0:
                mape = mean_absolute_percentage_error(
                    [p['actual'] for p in self.prediction_history[-5:]], 
                    [p['predicted'] for p in self.prediction_history[-5:]]
                ) * 100
                rmse = np.sqrt(mean_squared_error(
                    [p['actual'] for p in self.prediction_history[-5:]], 
                    [p['predicted'] for p in self.prediction_history[-5:]]
                ))
            else:
                mape = rmse = None
                
            return prediction, {'mape': mape, 'rmse': rmse}
            
        except Exception as e:
            print(f"Prediction error: {e}")
            return None, None

    def save_price_data(self, price_data):
        if price_data:
            # Save to file
            with open(self.output_file, 'a') as f:
                json.dump(price_data, f)
                f.write('\n')
            
            # Update price history
            self.price_history.append(price_data)
            
            # Make prediction for next interval
            next_prediction, metrics = self.make_prediction()
            
            if next_prediction is not None:
                self.prediction_history.append({
                    'timestamp': price_data['timestamp'],
                    'actual': price_data['price'],
                    'predicted': next_prediction
                })
                
                # Print current price and prediction metrics
                print(f"\n[{price_data['timestamp']}]")
                print(f"Current Price: ${price_data['price']:,.2f}")
                print(f"24h Change: {price_data['change_24h']:.2f}%")
                print(f"Volume: ${price_data['volume_24h']:,.0f}")
                if metrics['mape'] is not None:
                    print(f"Prediction Accuracy:")
                    print(f"  MAPE: {metrics['mape']:.2f}%")
                    print(f"  RMSE: ${metrics['rmse']:.2f}")
                print(f"Next Prediction: ${next_prediction:,.2f}")
                print("-" * 50)

    def run(self, duration_hours=24, interval_seconds=60):
        total_iterations = (duration_hours * 3600) // interval_seconds
        print(f"Starting Bitcoin price collection for {duration_hours} hours...")
        print(f"Data will be collected every {interval_seconds} seconds ({total_iterations} data points)")
        print("ARIMA model will take approximately 0.5-2 seconds to run after each data point")
        print("Press Ctrl+C to stop...")

        iteration = 0
        try:
            while iteration < total_iterations:
                price_data = self.fetch_bitcoin_price()
                self.save_price_data(price_data)
                iteration += 1
                time.sleep(interval_seconds)
                
        except KeyboardInterrupt:
            print("\nData collection stopped by user")
        except Exception as e:
            print(f"Unexpected error: {e}")
        finally:
            print(f"\nCollection complete. Data saved to {self.output_file}")
            print(f"Collected {iteration} data points")

if __name__ == "__main__":
    collector = BitcoinPriceCollector()
    # Collect data for 24 hours with 1-minute intervals
    collector.run(duration_hours=24, interval_seconds=60)

Starting Bitcoin price collection for 24 hours...
Data will be collected every 60 seconds (1440 data points)
Press Ctrl+C to stop...
[2025-05-16 21:10:54] Bitcoin Price: $103,238.00 | 24h Change: -0.49% | Volume: $26,115,628,336
[2025-05-16 21:11:54] Bitcoin Price: $103,235.00 | 24h Change: -0.50% | Volume: $26,114,012,789

Data collection stopped by user

Collection complete. Data saved to btc_prices.json
Collected 2 data points
